In [8]:
import requests
from bs4 import BeautifulSoup
import time
import os
import json

# --- Step 1: Automatically extract links ---
main_url = "https://www.flowermeaning.com/"
try:
    print(f"Fetching article links from {main_url}...")
    main_response = requests.get(main_url)
    main_soup = BeautifulSoup(main_response.text, "html.parser")
    urls = [a["href"] for a in main_soup.select("div.fusion-megamenu-title a") if a.get("href")]
    urls = list(dict.fromkeys(urls)) # Remove duplicates
    print(f"Found {len(urls)} flower links.")
except Exception as e:
    print(f"Error fetching main list: {e}")
    urls = []

# --- Step 2: Scrape and Clean ---
output_dir = "flower_texts"
data_dir = "flower_data"
for d in [output_dir, data_dir]:
    if not os.path.exists(d):
        os.makedirs(d)

flower_info = {}

for url in urls:
    flower_name = url.split('/')[-2]
    filename = f"{output_dir}/{flower_name}.txt"
    print(f"Processing: {flower_name}...")

    try:
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        content = soup.find("div", class_="post-content")

        # Extract image URL
        img_tag = soup.find("div", class_="fusion-text").find("img") if soup.find("div", class_="fusion-text") else None
        if not img_tag and content:
            img_tag = content.find("img")

        if img_tag:
            img_url = img_tag.get("data-orig-src") or img_tag.get("src")
            flower_info[flower_name] = img_url

        if content:
            for social in content.find_all(class_=["essb_links", "essb_displayed_postfloat"]):
                social.decompose()

            for ad_box in content.find_all("div", class_="code-block"):
                ad_box.decompose()

            ad_keywords = ["Click here"]
            for keyword in ad_keywords:
                for element in content.find_all(string=lambda text: keyword in text if text else False):
                    parent = element.find_parent(["div", "p", "section"])
                    if parent:
                        parent.decompose()

            text = content.get_text(separator="\n", strip=True)
            with open(filename, "w", encoding="utf-8") as f:
                f.write(text)

        time.sleep(1)
    except Exception as e:
        print(f"Error processing {url}: {e}")

# Save the image map to JSON in the root folder
with open("image_map.json", "w") as f:
    json.dump(flower_info, f, indent=4)

print("\nScraping finished. image_map.json created in the root directory.")

Fetching article links from https://www.flowermeaning.com/...
Found 98 flower links.
Processing: alstroemeria-flower-meaning...
Processing: amaryllis-flower-meaning...
Processing: anemone-flower-meaning...
Processing: anthurium-flower...
Processing: aster-flower-meaning...
Processing: azalea-flower-meaning...
Processing: baby-breath-flower...
Processing: begonia-flower...
Processing: bird-of-paradise-flower-meaning...
Processing: bleeding-heart-flower-meaning...
Processing: buttercup-flower-meaning...
Processing: cactus-flower-meaning...
Processing: calla-lily-meaning...
Processing: camellia-flower-meaning...
Processing: carnation-flower-meaning...
Processing: christmas-flowers...
Processing: chrysanthemum-flower-meaning...
Processing: columbine-flower-meaning...
Processing: crocus-flower-meaning...
Processing: daffodil-flower-meaning...
Processing: dahlia-flower-meaning...
Processing: daisy-flower-meaning...
Processing: dandelion-flower-meaning...
Processing: delphinium...
Processing:

In [12]:
import os
import requests
import json
from tqdm import tqdm

# --- Setup ---
image_map_path = "image_map.json"
images_dir = "flower_images"

if not os.path.exists(images_dir):
    os.makedirs(images_dir)

# Load the URLs we found in the previous step
try:
    with open(image_map_path, "r") as f:
        flower_info = json.load(f)
except FileNotFoundError:
    print("Error: image_map.json not found. Please run the scraper cell first.")
    flower_info = {}

# --- Download Process ---
print(f"Starting download of {len(flower_info)} images...")

for name, url in tqdm(flower_info.items()):
    if not url:
        continue

    # Determine file extension (usually .jpg)
    ext = url.split('.')[-1].split('?')[0] # Basic extension extractor
    if len(ext) > 4: ext = "jpg"

    filepath = os.path.join(images_dir, f"{name}.{ext}")

    if os.path.exists(filepath):
        continue # Skip if already downloaded

    try:
        img_data = requests.get(url, timeout=10).content
        with open(filepath, "wb") as handler:
            handler.write(img_data)
    except Exception as e:
        print(f"Could not download {name}: {e}")

print(f"\nFinished! Images are saved in the '{images_dir}' folder.")

Starting download of 97 images...


100%|████████████████████████████████████████████████████████████████████████████████| 97/97 [00:00<00:00, 88253.25it/s]


Finished! Images are saved in the 'flower_images' folder.
